# Viola-Jones cascade: cat detection

```text
Viola-Jones (CVPR 2001) "Rapid Object Detection using a Boosted Cascade of Simple Features",
implemented from scratch and evaluated as an object DETECTION task (cats) following the paper's protocol.

How the cat/dog photos are turned into a detection problem
  * "object"      = a whole cat photo, squashed to the 24x24 base window (the analogue of the paper's aligned
                    face crops); mirrored images are added as in the paper.
  * "non-object"  = sub-windows (every position, every scale) of DOG photos that have been "messed up" with a random
                    warp (mirror, rotation, zoom, stretch, shear, off-centre crop). Later cascade stages train on the
                    dog windows that survive the earlier stages (the paper scans non-face images for false positives).
  * test          = held-out cats as positive windows; held-out dog photos, warped, scanned at all scales/positions as
                    negatives. Reported as detection rate vs false-positive RATE (false positives / windows scanned,
                    as in the paper's ROC), plus seconds per scanned photo and features evaluated per window.
  * train / validation / test = disjoint photos. Validation photos tune each stage's threshold.

Paper components implemented directly (numpy / scipy only)
  rectangle features (2-, 3-, 4-rect; all 162,336 enumerated at 24x24, a random subset is trained on),
  integral image + second (squared) integral image for variance normalisation, AdaBoost exactly as Table 1,
  attentional cascade with validation-tuned stage thresholds, scaling the DETECTOR (not the image) to scan
  multiple scales (factor 1.25, step size delta), ROC from sweeping the last stage's threshold / removing layers.

Baselines (stand-ins for the systems in the paper's comparison, all scanned over the same windows)
  MLP on pixels (~Rowley et al.), naive Bayes on quantised Haar-wavelet coefficients (~Schneiderman-Kanade),
  Winnow on sparse binary pixel features (~Roth et al. SNoW), and a single 200-feature AdaBoost classifier
  (paper Sec. 3.2: what the cascade is meant to speed up).
```

## How to run this notebook
* Put the two class folders next to the notebook (or change `data_dir`, `pos_folder` and `neg_folder` in the configuration cell) and run the cells top to bottom. The defaults are `Cat` (positive) and `Dog` (negative).
* Every option that used to be a command-line flag is now a field of the `a` object in the **Configuration** cell.
* Image loading uses threads instead of processes, because process pools do not work with functions defined in a notebook on macOS.
* The defaults match the earlier script (about an hour to train on one core). For a quick test try `n_per_class=1500, n_features=8000, neg_images=30, baseline_images=15, max_stages=8`.

## Imports and helpers

In [ ]:
import json
import os
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from types import SimpleNamespace

import numpy as np
import scipy.sparse as sp
from numpy.lib.stride_tricks import sliding_window_view
from PIL import Image

warnings.filterwarnings("ignore")
T0 = time.perf_counter()
B = 24  # base resolution of the detector (paper: 24x24)


def log(*a):
    print(f"[{time.perf_counter() - T0:7.1f}s]", *a, flush=True)

## 1. Data

In [ ]:
def find_images(root, pos_folder="Cat", neg_folder="Dog"):
    """image paths of the positive (object) and negative classes; folders are matched by name, case-insensitive"""
    pos, neg = [], []
    for dp, _, fns in os.walk(root):
        name = os.path.basename(dp).lower()
        if name not in (pos_folder.lower(), neg_folder.lower()):
            continue
        tgt = pos if name == pos_folder.lower() else neg
        tgt += [os.path.join(dp, f) for f in fns if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    return sorted(pos), sorted(neg)


def _load_one(args):
    path, w, h = args
    try:
        im = Image.open(path)
        im.draft("L", (w * 2, h * 2))
        im = im.convert("L").resize((w, h), Image.LANCZOS)
        return np.asarray(im, dtype=np.uint8)
    except Exception:
        return None  # a few files in this dataset are corrupt / empty


def load_images(paths, n, w, h, rng, workers):
    paths = [paths[i] for i in rng.permutation(len(paths))][: int(n * 1.05) + 20]
    with ThreadPoolExecutor(workers) as ex:
        out = list(ex.map(_load_one, [(p, w, h) for p in paths], chunksize=32))
    return np.stack([o for o in out if o is not None][:n])


def to_base(arr):
    return np.stack([np.asarray(Image.fromarray(x).resize((B, B), Image.LANCZOS)) for x in arr])

## 2. Rectangle features + integral image  (paper Sec. 2)

In [ ]:
# Weights follow the paper's text: two-rect = difference of the two sums; three-rect = centre minus the two
# outside rectangles; four-rect = difference between diagonal pairs.
FTYPES = [
    ((2, 1), [(0, 0, +1), (1, 0, -1)]),
    ((1, 2), [(0, 0, +1), (0, 1, -1)]),
    ((3, 1), [(0, 0, -1), (1, 0, +1), (2, 0, -1)]),
    ((1, 3), [(0, 0, -1), (0, 1, +1), (0, 2, -1)]),
    ((2, 2), [(0, 0, +1), (1, 0, -1), (0, 1, -1), (1, 1, +1)]),
]


def enumerate_feature_params(base=B):
    P = []
    for t, ((bw, bh), _) in enumerate(FTYPES):
        for cw in range(1, base // bw + 1):
            for ch in range(1, base // bh + 1):
                W, H = bw * cw, bh * ch
                xs, ys = np.meshgrid(np.arange(base - W + 1), np.arange(base - H + 1), indexing="ij")
                n = xs.size
                P.append(np.stack([np.full(n, t), np.full(n, cw), np.full(n, ch), xs.ravel(), ys.ravel()], 1))
    return np.concatenate(P).astype(np.int32)


def build_feature_matrix(params, base=B):
    """Sparse (F x (base+1)^2): feature_values = M @ flattened_integral_image (shared corners merge, leaving the
    6 / 8 / 9 array references of the paper for 2 / 3 / 4-rectangle features)."""
    S = base + 1
    rows, cols, vals = [], [], []
    for f, (t, cw, ch, x, y) in enumerate(params):
        for dx, dy, wt in FTYPES[t][1]:
            x0, y0 = x + dx * cw, y + dy * ch
            x1, y1 = x0 + cw, y0 + ch
            for yy, xx, s in ((y1, x1, 1), (y0, x1, -1), (y1, x0, -1), (y0, x0, 1)):
                rows.append(f), cols.append(yy * S + xx), vals.append(wt * s)
    M = sp.csr_matrix((vals, (rows, cols)), shape=(len(params), S * S), dtype=np.float32)
    M.eliminate_zeros()
    return M


def integral(imgs):
    """imgs (N,24,24) uint8 -> (flattened integral images (N,625) float32, sigma (N,)) for training windows."""
    if len(imgs) > 1000:
        parts = [integral(imgs[i:i + 1000]) for i in range(0, len(imgs), 1000)]
        return np.concatenate([p[0] for p in parts]), np.concatenate([p[1] for p in parts])
    a = imgs.astype(np.float64)
    N, Bs = a.shape[0], a.shape[1]
    ii = np.zeros((N, Bs + 1, Bs + 1))
    ii[:, 1:, 1:] = a.cumsum(1).cumsum(2)
    ii2_corner = (a * a).cumsum(1).cumsum(2)[:, -1, -1]  # integral image of the squared window
    n = Bs * Bs
    mean = ii[:, -1, -1] / n
    sigma = np.sqrt(np.maximum(ii2_corner / n - mean ** 2, 1.0))
    return ii.reshape(N, -1).astype(np.float32), sigma.astype(np.float32)

## 3. AdaBoost with single-feature weak learners (Table 1) and the cascade (Sec. 4)

In [ ]:
class Stage:
    """strong classifier: margin = sum_t alpha_t h_t(x) - 0.5 sum_t alpha_t, h_t = [p_t f_t(x) < p_t thr_t];
    passes if margin >= theta (theta = 0 is the default AdaBoost threshold; lowered by validation tuning)."""

    def __init__(self, M, feat_idx, thr, par, alpha, theta):
        self.M, self.feat_idx = M, np.asarray(feat_idx)
        self.thr, self.par = np.asarray(thr, np.float32), np.asarray(par, np.float32)
        self.alpha, self.theta = np.asarray(alpha, np.float32), float(theta)
        self.asum = float(self.alpha.sum())
        S = int(round(np.sqrt(M.shape[1])))
        k, R = M.shape[0], max(1, int(np.diff(M.indptr).max()))
        self.cy, self.cx = np.zeros((k, R)), np.zeros((k, R))  # corner coordinates of every feature (padded)
        self.cc = np.zeros((k, R), np.float32)  # corner coefficients (0 for padding)
        for i in range(k):
            lo, hi = M.indptr[i], M.indptr[i + 1]
            c = M.indices[lo:hi]
            self.cy[i, :hi - lo], self.cx[i, :hi - lo], self.cc[i, :hi - lo] = c // S, c % S, M.data[lo:hi]

    def margin(self, ii, sig):  # on 24x24 training windows
        F = (self.M @ ii.T).T / sig[:, None]
        h = (self.par * F < self.par * self.thr)
        return h.astype(np.float32) @ self.alpha - 0.5 * self.asum


def prepare_chunks(Mall, ii, sig, y, chunk=2500):
    """per-feature sort order of the training pool (computed once per stage, as in the paper)"""
    chunks = []
    for s in range(0, Mall.shape[0], chunk):
        Xt = ((Mall[s:s + chunk] @ ii.T) / sig[None, :]).astype(np.float32)
        order = np.argsort(Xt, axis=1).astype(np.int32)
        Xs = np.take_along_axis(Xt, order, 1)
        chunks.append(dict(start=s, order=order, ys=y[order], tie=Xs[:, 1:] == Xs[:, :-1]))
        del Xt, Xs
    return chunks


def best_weak(chunks, w32):
    """Table 1, steps 2-3: for every feature the optimal threshold + parity (O(N) via cumulative weights);
    return the single feature with the lowest weighted error."""
    best = (np.inf, None)
    for c in chunks:
        ws = w32[c["order"]]
        cs = np.cumsum(ws, 1)
        sp_ = np.cumsum(ws * c["ys"], 1)
        sn = cs - sp_
        Tp, Tn = sp_[:, -1:], sn[:, -1:]
        e_pos_left = sn + (Tp - sp_)  # parity +1: h = 1 if f < thr
        e_neg_left = sp_ + (Tn - sn)  # parity -1: h = 1 if f > thr
        e = np.minimum(e_pos_left, e_neg_left)[:, :-1]
        e[c["tie"]] = np.inf
        j, k = np.unravel_index(np.argmin(e), e.shape)
        if e[j, k] < best[0]:
            best = (float(e[j, k]), (c, int(j), int(k), 1 if e_pos_left[j, k] <= e_neg_left[j, k] else -1))
    return best[1]


def tune_theta(margin, y, d_min):
    """largest threshold that still keeps >= d_min of the (validation) positives"""
    pm = np.sort(margin[y == 1])
    theta = pm[int(np.floor((1 - d_min) * len(pm)))] - 1e-6
    return theta, float((margin[y == 0] >= theta).mean())

In [ ]:
def train_stage(Mall, ii, sig, y, cap, tune=None, d_min=0.99, f_max=0.5):
    """One boosted classifier (Table 1). With tune=(ii_v, sig_v, y_v): add features until the validation
    detection rate >= d_min with false-positive rate <= f_max, lowering the threshold to keep d_min.
    Without tune: exactly `cap` rounds at the default threshold."""
    y = y.astype(np.uint8)
    m, l = int((y == 0).sum()), int(y.sum())
    w = np.where(y == 1, 1.0 / (2 * l), 1.0 / (2 * m))
    chunks = prepare_chunks(Mall, ii, sig, y)
    feats, thr, par, alpha = [], [], [], []
    if tune is not None:
        ii_v, sig_v, y_v = tune
        v_raw = np.zeros(len(y_v))
    theta, fpr = 0.0, None
    for t in range(cap):
        w = w / w.sum()
        c, j, k, p = best_weak(chunks, w.astype(np.float32))
        g = c["start"] + j
        Xj = (np.asarray(Mall[g] @ ii.T).ravel() / sig).astype(np.float32)
        srt = np.sort(Xj)
        a, b = srt[k], srt[k + 1]
        th = np.float32((float(a) + float(b)) / 2)
        if p == 1 and not (a < th <= b):
            th = b
        if p == -1 and not (a <= th < b):
            th = a
        h = (p * Xj < p * th).astype(np.uint8)
        eps = min(max(float((w * (h != y)).sum()), 1e-9), 1 - 1e-9)
        beta = eps / (1 - eps)
        w = w * np.where(h == y, beta, 1.0)
        feats.append(g), thr.append(th), par.append(p), alpha.append(np.log(1 / beta))
        if tune is not None:
            Fv = np.asarray(Mall[g] @ ii_v.T).ravel() / sig_v
            v_raw += alpha[-1] * (p * Fv < p * th)
            theta, fpr = tune_theta(v_raw - 0.5 * float(np.sum(alpha)), y_v, d_min)
            if fpr <= f_max:
                break
    del chunks
    return Stage(Mall[feats], feats, thr, par, alpha, theta), fpr


def stage_cap(s, max_feats):
    return min(max_feats, 10 * 2 ** s)  # 10, 20, 40, 80, ... (paper's stages also grow)

## 4. Scanning: scaled detector, multiple scales and locations (paper Sec. 5)

In [ ]:
def build_windows(W, H, scale0, factor, delta):
    """Scales scale0 * factor^k; at scale s the window is round(24 s) wide and is shifted by round(s * delta)
    pixels (the paper's scheme). jx/jy are the matching positions in an image pyramid (used by baselines)."""
    wins, s = [], scale0
    while B * s <= min(W, H):
        Wp, step = int(round(B * s)), max(1, int(round(s * delta)))
        x, y = np.arange(0, W - Wp + 1, step), np.arange(0, H - Wp + 1, step)
        Ws, Hs = int(round(W / s)), int(round(H / s))
        if Ws >= B and Hs >= B and len(x) and len(y):
            jx = np.clip(np.rint(x * Ws / W), 0, Ws - B).astype(int)
            jy = np.clip(np.rint(y * Hs / H), 0, Hs - B).astype(int)
            wins.append(dict(s=s, Wp=Wp, x=x, y=y, Ws=Ws, Hs=Hs, jx=jx, jy=jy))
        s *= factor
    return wins


def window_meta(wins):
    xs, ys, ws = [], [], []
    for w in wins:
        yy, xx = np.meshgrid(w["y"], w["x"], indexing="ij")
        xs.append(xx.ravel()), ys.append(yy.ravel()), ws.append(np.full(xx.size, w["Wp"]))
    return np.concatenate(xs).astype(np.int32), np.concatenate(ys).astype(np.int32), np.concatenate(ws).astype(np.int32)


def scene_integrals(img):
    a = img.astype(np.float64)
    H, W = a.shape
    ii, ii2 = np.zeros((H + 1, W + 1)), np.zeros((H + 1, W + 1))
    ii[1:, 1:] = a.cumsum(0).cumsum(1)
    ii2[1:, 1:] = (a * a).cumsum(0).cumsum(1)
    return ii, ii2


def stage_margin_scan(st, iif, S1, pos, f, sig):
    """margins of stage `st` for windows with top-left flat index `pos`, detector scaled by f = Wp/24:
    corner coordinates are scaled and rounded, the feature value is divided by f^2 (area) and by sigma."""
    off = np.rint(f * st.cy).astype(np.int64) * S1 + np.rint(f * st.cx).astype(np.int64)  # (k, R)
    k, R = off.shape
    ch = max(1, 4_000_000 // (k * R))
    out = np.empty(len(pos), np.float32)
    for s in range(0, len(pos), ch):
        p = pos[s:s + ch]
        v = iif[p[:, None, None] + off[None]]  # (n, k, R)
        F = (v * st.cc[None]).sum(2) / (f * f * sig[s:s + ch, None])
        h = (st.par * F < st.par * st.thr)
        out[s:s + ch] = h.astype(np.float32) @ st.alpha - 0.5 * st.asum
    return out


def cascade_scan(stages, ii, ii2, wins, W, H):
    """Run the cascade (with early rejection) over every window of one scene.
    Returns (key, depth, n_features_evaluated). depth = number of stages passed. key is a scalar ranking:
    rejected at stage k -> k + 0.5 + normalised margin in [k, k+1); passed all K stages -> K + 0.5 + margin >= K.
    Thresholding `key` reproduces the paper's ROC construction (remove layers / sweep the last layer's threshold)."""
    S1, iif, ii2f = W + 1, ii.ravel(), ii2.ravel()
    K, nfeat, keys, depths = len(stages), 0, [], []
    for w in wins:
        Wp = w["Wp"]
        f = Wp / B
        yy, xx = np.meshgrid(w["y"], w["x"], indexing="ij")
        pos = (yy.ravel() * S1 + xx.ravel()).astype(np.int64)
        d = Wp * S1
        s1 = iif[pos + d + Wp] - iif[pos + Wp] - iif[pos + d] + iif[pos]
        s2 = ii2f[pos + d + Wp] - ii2f[pos + Wp] - ii2f[pos + d] + ii2f[pos]
        n = Wp * Wp
        mean = s1 / n
        sig = np.sqrt(np.maximum(s2 / n - mean * mean, 1.0))
        key = np.full(len(pos), 0.5, np.float32)
        depth = np.zeros(len(pos), np.int16)
        alive = np.arange(len(pos))
        mn_alive = np.zeros(len(pos), np.float32)
        for k, st in enumerate(stages):
            if alive.size == 0:
                break
            m = stage_margin_scan(st, iif, S1, pos[alive], f, sig[alive])
            nfeat += alive.size * len(st.thr)
            mn = m / st.asum
            ok = m >= st.theta
            key[alive[~ok]] = k + 0.5 + mn[~ok]
            depth[alive[~ok]] = k
            alive, mn_alive = alive[ok], mn[ok]
        key[alive] = K + 0.5 + mn_alive if K else 0.5
        depth[alive] = K
        keys.append(key), depths.append(depth)
    return np.concatenate(keys), np.concatenate(depths), nfeat

In [ ]:
def crop24(img, x, y, w):
    return np.asarray(Image.fromarray(np.ascontiguousarray(img[y:y + w, x:x + w])).resize((B, B), Image.LANCZOS))


def collect_negatives(stages, images, wins, meta, n_want, per_image, max_images, rng):
    """Scan (warped) dog photos with the cascade built so far and collect windows that still pass (false positives).
    With no stages yet this is a random sample of sub-windows (the paper's first stage)."""
    xs, ys, ws = meta
    out, got = [], 0
    for i in rng.permutation(len(images))[:max_images]:
        img = warp_dog(images[i], rng)  # 'mess up' the dog photo
        if stages:
            ii, ii2 = scene_integrals(img)
            _, depth, _ = cascade_scan(stages, ii, ii2, wins, img.shape[1], img.shape[0])
            surv = np.flatnonzero(depth == len(stages))
        else:
            surv = np.arange(len(xs))
        if surv.size == 0:
            continue
        pick = rng.choice(surv, min(per_image, surv.size), replace=False)
        out += [crop24(img, xs[j], ys[j], ws[j]) for j in pick]
        got += len(pick)
        if got >= n_want:
            break
    return np.stack(out) if out else np.zeros((0, B, B), np.uint8)


def train_cascade(Mall, P, V, train_dogs, val_dogs, wins_c, meta_c, a, rng, all_neg):
    """P / V = (integral images, sigma) of train / validation cat windows. Negatives for every stage are
    collected fresh from the dog photos by scanning the cascade built so far."""
    stages, F = [], 1.0
    pos_alive, vpos_alive = np.arange(len(P[0])), np.arange(len(V[0]))
    for s in range(a.max_stages):
        t0 = time.perf_counter()
        neg = collect_negatives(stages, train_dogs, wins_c, meta_c, a.max_neg, a.per_image, a.collect_images, rng)
        vneg = collect_negatives(stages, val_dogs, wins_c, meta_c, a.val_neg, a.per_image, a.collect_images, rng)
        if len(neg) < a.min_train_neg or len(vneg) < a.min_val_neg or len(vpos_alive) < a.min_val_pos:
            log(f"stop: too few negatives/positives left (train neg {len(neg)}, val neg {len(vneg)}, val pos {len(vpos_alive)})")
            break
        all_neg.append(neg)
        Nii, Vii = integral(neg), integral(vneg)
        psel = pos_alive if len(pos_alive) <= a.max_pos else rng.choice(pos_alive, a.max_pos, replace=False)
        ii = np.concatenate([P[0][psel], Nii[0]])
        sig = np.concatenate([P[1][psel], Nii[1]])
        y = np.r_[np.ones(len(psel)), np.zeros(len(neg))].astype(np.uint8)
        vii = np.concatenate([V[0][vpos_alive], Vii[0]])
        vsig = np.concatenate([V[1][vpos_alive], Vii[1]])
        vy = np.r_[np.ones(len(vpos_alive)), np.zeros(len(vneg))].astype(np.uint8)
        st, fpr = train_stage(Mall, ii, sig, y, stage_cap(s, a.max_stage_features), tune=(vii, vsig, vy),
                              d_min=a.d_min, f_max=a.f_max)
        if fpr > 0.95:
            log(f"stage {s + 1}: no progress (val FPR {fpr:.2f} with {len(st.thr)} features) -> stop")
            break
        stages.append(st)
        pos_alive = pos_alive[st.margin(P[0][pos_alive], P[1][pos_alive]) >= st.theta]
        vpos_alive = vpos_alive[st.margin(V[0][vpos_alive], V[1][vpos_alive]) >= st.theta]
        F *= fpr
        log(f"stage {s + 1:2d}: {len(st.thr):3d} features ({time.perf_counter() - t0:5.1f}s) | stage val FPR {fpr:.3f} | "
            f"cascade est. FPR {F:.2e} | cascade val TPR {len(vpos_alive) / len(V[0]):.3f} | "
            f"train pos {len(pos_alive)}, neg {len(neg)}")
        if F <= a.f_target:
            break
    return stages

## 5. Warped dog photos as negatives; evaluation (detection rate vs false-positive rate)

In [ ]:
def warp_dog(img, rng):
    """'Mess up' a dog photo: random mirror, rotation, zoom, anisotropic stretch, shear and off-centre crop.
    The photo is reflect-padded first so no blank corners appear."""
    H, W = img.shape
    p = int(1.5 * max(H, W))
    big = Image.fromarray(np.pad(img, p, mode="reflect"))
    if rng.random() < 0.5:
        big = big.transpose(Image.FLIP_LEFT_RIGHT)
    th = np.deg2rad(rng.uniform(-30, 30))
    s = np.exp(rng.uniform(np.log(0.5), np.log(1.5)))  # input pixels per output pixel (zoom)
    asp = np.exp(rng.uniform(np.log(0.75), np.log(1.33)))  # stretch
    sh = rng.uniform(-0.25, 0.25)  # shear
    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
    A = R @ np.array([[s * asp, sh * s / asp], [0.0, s / asp]])
    cx = p + W / 2 + rng.uniform(-0.25, 0.25) * W
    cy = p + H / 2 + rng.uniform(-0.25, 0.25) * H
    c = cx - A[0, 0] * W / 2 - A[0, 1] * H / 2
    f = cy - A[1, 0] * W / 2 - A[1, 1] * H / 2
    out = big.transform((W, H), Image.AFFINE, (A[0, 0], A[0, 1], c, A[1, 0], A[1, 1], f), resample=Image.BILINEAR)
    return np.asarray(out, np.uint8)


def keys_24(stages, ii, sig):
    """same ranking key as cascade_scan, for 24x24 windows (the test cats)"""
    K = len(stages)
    key = np.full(len(ii), 0.5, np.float32)
    alive, mn_alive = np.arange(len(ii)), np.zeros(len(ii), np.float32)
    for k, st in enumerate(stages):
        if alive.size == 0:
            break
        m = st.margin(ii[alive], sig[alive])
        mn, ok = m / st.asum, m >= st.theta
        key[alive[~ok]] = k + 0.5 + mn[~ok]
        alive, mn_alive = alive[ok], mn[ok]
    key[alive] = K + 0.5 + mn_alive
    return key


def roc_from_scores(pos, neg):
    """detection rate vs false-positive RATE (false positives / windows scanned, as in the paper's ROC)"""
    ps, ns = np.sort(pos), np.sort(neg)
    ths = np.unique(ps)
    tpr = 1 - np.searchsorted(ps, ths, "left") / len(ps)
    fpr = 1 - np.searchsorted(ns, ths, "left") / len(ns)
    o = np.argsort(fpr, kind="stable")
    return fpr[o], np.maximum.accumulate(tpr[o])


def standardize(imgs):
    x = imgs.reshape(len(imgs), -1).astype(np.float32)
    return (x - x.mean(1, keepdims=True)) / np.maximum(x.std(1, keepdims=True), 1.0)

## 6. Baselines (stand-ins for the paper's comparison systems)

In [ ]:
def haar_coeffs(x):
    n = int(np.sqrt(x.shape[1]))
    x = x.reshape(-1, n, n)

    def h2(a):
        p, q, r, s = a[:, 0::2, 0::2], a[:, 0::2, 1::2], a[:, 1::2, 0::2], a[:, 1::2, 1::2]
        return (p + q + r + s) / 4, (p - q + r - s) / 4, (p + q - r - s) / 4, (p - q - r + s) / 4
    ll1, a1, b1, c1 = h2(x)
    ll2, a2, b2, c2 = h2(ll1)
    return np.concatenate([t.reshape(len(x), -1) for t in (a1, b1, c1, ll2, a2, b2, c2)], 1)


class Winnow:
    """Winnow2 (the learning rule in SNoW) on sparse binary features: one active feature per pixel."""

    def __init__(self, alpha=1.2, epochs=8, nbins=8, seed=0):
        self.alpha, self.epochs, self.nbins, self.seed = alpha, epochs, nbins, seed

    def _idx(self, x):
        b = np.clip(((x + 2.5) / 5 * self.nbins).astype(int), 0, self.nbins - 1)
        return b + np.arange(x.shape[1]) * self.nbins

    def fit(self, x, y):
        idx = self._idx(x)
        self.theta = float(x.shape[1])
        self.w = np.ones(x.shape[1] * self.nbins)
        rng = np.random.default_rng(self.seed)
        for _ in range(self.epochs):
            for i in rng.permutation(len(y)):
                s = self.w[idx[i]].sum()
                if y[i] == 1 and s < self.theta:
                    self.w[idx[i]] *= self.alpha
                elif y[i] == 0 and s >= self.theta:
                    self.w[idx[i]] /= self.alpha
            np.clip(self.w, 1e-6, 1e6, out=self.w)
        return self

    def score(self, x):
        return self.w[self._idx(x)].sum(1) - self.theta


def extract_patches(img, wins):
    """24x24 patches for every window, from an image pyramid (same window order as window_meta)"""
    out = []
    for w in wins:
        a = np.asarray(Image.fromarray(img).resize((w["Ws"], w["Hs"]), Image.LANCZOS))
        out.append(sliding_window_view(a, (B, B))[np.ix_(w["jy"], w["jx"])].reshape(-1, B, B))
    return np.concatenate(out)


def batched(fn, X, bs=40000):
    return np.concatenate([fn(X[i:i + bs]) for i in range(0, len(X), bs)])

## 7. Plots

In [ ]:
def make_plots(out_dir, curves, speed_rows):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
    for name, (fpr, tpr) in curves.items():
        ax[0].plot(np.maximum(fpr, 1e-8), tpr, label=name, lw=2 if name.startswith("Viola") else 1.2)
    ax[0].set(xscale="log", xlabel="false-positive rate (false positives / windows scanned)",
              ylabel="detection rate (cats found)", title="ROC on held-out cats vs warped held-out dog photos")
    ax[0].legend(fontsize=7)
    ax[1].barh([r["name"] for r in speed_rows], [r["sec_per_image"] for r in speed_rows])
    ax[1].set(xscale="log", xlabel="seconds to scan one photo (log)", title="Scan time")
    ax[1].invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "results.png"), dpi=130)
    plt.close()

## Configuration
These were the command-line flags. Edit them here.

In [ ]:
a = SimpleNamespace(
    data_dir=".",                # folder that contains the two class folders
    pos_folder="Cat",            # name of the folder with the positive class (the object to detect)
    neg_folder="Dog",            # name of the folder with the negative class (used for the non-object windows)
    out="vj_detection_results",  # results folder (also holds the image cache)
    n_per_class=12500, train_frac=0.6, val_frac=0.2,   # the rest is the test split
    img_w=192, img_h=144,        # dog photos are resized to this size (paper's test images: 384x288)
    neg_images=130,              # warped test dog photos scanned for false positives (paper: 130 images)
    # --- cascade training ---
    n_features=20000,            # random subset of the 162,336 features (memory!)
    d_min=0.99, f_max=0.5,       # per-stage detection / false-positive targets (validation set)
    f_target=1e-5,               # stop when the estimated cascade false-positive rate falls below this
    max_stages=25, max_stage_features=150,
    max_pos=3000, max_neg=5000, val_neg=3000,
    per_image=50, collect_images=1500,
    min_train_neg=300, min_val_neg=50, min_val_pos=50,
    mono_features=200,           # size of the single (non-cascade) AdaBoost baseline
    # --- scanning: ROC run (paper Fig. 6: scale 1.0, step 1.0); speed run (scale 1.25, step 1.5) ---
    scale0=1.0, factor=1.25, delta=1.0,
    speed_scale0=1.25, speed_delta=1.5, speed_images=20,
    baseline_images=40,          # photos on which all methods are compared
    skip_baselines=False,
    workers=max(1, (os.cpu_count() or 2) - 1),
    seed=0,
)
os.makedirs(a.out, exist_ok=True)
rng = np.random.default_rng(a.seed)
W, H = a.img_w, a.img_h

### Load the data and split train / validation / test

In [ ]:
# ---- data -----------------------------------------------------------------------------------
cat_paths, dog_paths = find_images(a.data_dir, a.pos_folder, a.neg_folder)
log(f"found {len(cat_paths)} cat / {len(dog_paths)} dog files in {os.path.abspath(a.data_dir)}")
cache = os.path.join(a.out, f"cache_n{a.n_per_class}_{W}x{H}_seed{a.seed}.npz")
if os.path.exists(cache):
    z = np.load(cache)
    cats64, dogs = z["cats"], z["dogs"]
else:
    n = min(a.n_per_class, len(cat_paths), len(dog_paths))
    cats64 = load_images(cat_paths, n, 64, 64, rng, a.workers)
    dogs = load_images(dog_paths, n, W, H, rng, a.workers)
    np.savez_compressed(cache, cats=cats64, dogs=dogs)
n = min(len(cats64), len(dogs))
cats64, dogs = cats64[:n], dogs[:n]
ntr, nva = int(a.train_frac * n), int(a.val_frac * n)
tr, va, te = slice(0, ntr), slice(ntr, ntr + nva), slice(ntr + nva, n)
log(f"{n} cats + {n} dogs -> train {ntr}, val {nva}, test {n - ntr - nva} photos per class; dog photos {W}x{H}")

cat_tr = to_base(cats64[tr])
pos_win = np.concatenate([cat_tr, cat_tr[:, :, ::-1]])  # cats + mirror images (as for faces in the paper)
cat_va = to_base(cats64[va])
P, V = integral(pos_win), integral(cat_va)
train_dogs, val_dogs = dogs[tr], dogs[va]

### Rectangle features

In [ ]:
# ---- features -------------------------------------------------------------------------------
allp = enumerate_feature_params()
log(f"exhaustive rectangle feature set at {B}x{B}: {len(allp):,} (paper: 'over 180,000')")
params = allp[rng.choice(len(allp), min(a.n_features, len(allp)), replace=False)]
Mall = build_feature_matrix(params)
log(f"training on a random subset of {len(params):,} features")

### Train the cascade

In [ ]:
# ---- train cascade --------------------------------------------------------------------------
wins_c = build_windows(W, H, 1.0, 1.25, 1.5)
meta_c = window_meta(wins_c)
all_neg = []
t0 = time.perf_counter()
stages = train_cascade(Mall, P, V, train_dogs, val_dogs, wins_c, meta_c, a, rng, all_neg)
t_casc = time.perf_counter() - t0
assert stages, "no stages trained - relax --d-min / --f-max"
sizes = [len(s.thr) for s in stages]
K = len(stages)
log(f"CASCADE: {K} stages, features per stage {sizes}, total {sum(sizes)}; trained in {t_casc:.0f}s")

### Single boosted classifier (paper Sec. 3.2)

In [ ]:
# ---- single boosted classifier (paper Sec. 3.2) ---------------------------------------------
negs = np.concatenate(all_neg)
nsel = rng.choice(len(negs), min(a.max_neg, len(negs)), replace=False)
Nii = integral(negs[nsel])
psel = rng.choice(len(P[0]), min(a.max_pos, len(P[0])), replace=False)
t0 = time.perf_counter()
mono, _ = train_stage(Mall, np.concatenate([P[0][psel], Nii[0]]), np.concatenate([P[1][psel], Nii[1]]),
                      np.r_[np.ones(len(psel)), np.zeros(len(nsel))].astype(np.uint8), a.mono_features)
t_mono = time.perf_counter() - t0
log(f"single AdaBoost classifier ({a.mono_features} features) trained in {t_mono:.0f}s")

### Baselines

In [ ]:
# ---- baselines ------------------------------------------------------------------------------
baselines = {}  # name -> (score_fn on standardized (n,576) windows, native threshold, fit seconds)
if not a.skip_baselines:
    from sklearn.naive_bayes import CategoricalNB
    from sklearn.neural_network import MLPClassifier
    from sklearn.preprocessing import KBinsDiscretizer
    kb_ = min(len(pos_win), len(negs), 2 * a.max_pos)
    Xtr = standardize(np.concatenate([pos_win[rng.choice(len(pos_win), kb_, replace=False)],
                                      negs[rng.choice(len(negs), kb_, replace=False)]]))
    ytr = np.r_[np.ones(kb_), np.zeros(kb_)].astype(np.uint8)
    vneg = collect_negatives([], val_dogs, wins_c, meta_c, a.val_neg, a.per_image, a.collect_images, rng)
    Xva = standardize(np.concatenate([cat_va, vneg]))
    yva = np.r_[np.ones(len(cat_va)), np.zeros(len(vneg))].astype(np.uint8)

    best = None  # Rowley et al. stand-in
    for hid in [(32,), (128,)]:
        t0 = time.perf_counter()
        m = MLPClassifier(hid, alpha=1e-3, max_iter=200, early_stopping=True, random_state=a.seed).fit(Xtr, ytr)
        acc = (m.predict(Xva) == yva).mean()
        log(f"  MLP {hid}: val acc {acc:.3f}")
        if best is None or acc > best[0]:
            best = (acc, m, time.perf_counter() - t0, hid)
    _, mlp, tf, hid = best
    baselines[f"MLP on pixels {hid} (~Rowley)"] = (lambda x, m=mlp: m.predict_proba(x)[:, 1], 0.5, tf)

    t0 = time.perf_counter()  # Schneiderman-Kanade stand-in
    kbd = KBinsDiscretizer(5, encode="ordinal", strategy="quantile").fit(haar_coeffs(Xtr))
    nbm = CategoricalNB(alpha=1.0, min_categories=5).fit(kbd.transform(haar_coeffs(Xtr)).astype(int), ytr)
    tf = time.perf_counter() - t0

    def nb_score(x):
        lp = nbm.predict_log_proba(kbd.transform(haar_coeffs(x)).astype(int))
        return lp[:, 1] - lp[:, 0]
    baselines["Wavelet naive-Bayes (~Schneiderman)"] = (nb_score, 0.0, tf)

    best = None  # Roth et al. stand-in
    for al in (1.05, 1.2):
        t0 = time.perf_counter()
        wm = Winnow(alpha=al).fit(Xtr, ytr)
        acc = ((wm.score(Xva) >= 0) == yva).mean()
        log(f"  Winnow alpha={al}: val acc {acc:.3f}")
        if best is None or acc > best[0]:
            best = (acc, wm, time.perf_counter() - t0)
    baselines["Winnow/SNoW-style (~Roth)"] = (best[1].score, 0.0, best[2])

### Test data

In [ ]:
# ---- test data: held-out cats (positives) and warped held-out dog photos (negatives) ---------
cat_te = to_base(cats64[te])
Pte, Xpos = integral(cat_te), standardize(cat_te)
wrng = np.random.default_rng(a.seed + 1)
test_dogs = dogs[te]
neg_imgs = [warp_dog(test_dogs[i % len(test_dogs)], wrng) for i in range(a.neg_images)]
ni, nb = len(neg_imgs), min(a.baseline_images, a.neg_images)
wins = build_windows(W, H, a.scale0, a.factor, a.delta)
nwin = sum(len(w["x"]) * len(w["y"]) for w in wins)
log(f"test: {len(cat_te)} cat windows; {ni} warped dog photos x {nwin:,} windows = {ni * nwin / 1e6:.1f}M negative windows")

### Scan the warped dog photos with every method

In [ ]:
# ---- scan the warped dog photos with every method --------------------------------------------
neg_vj, nfeat = np.zeros((ni, nwin), np.float32), 0
for i, img in enumerate(neg_imgs):
    ii, ii2 = scene_integrals(img)
    neg_vj[i], _, nf = cascade_scan(stages, ii, ii2, wins, W, H)
    nfeat += nf
log(f"cascade evaluates {nfeat / (ni * nwin):.2f} features per window on average "
    f"(of {sum(sizes)} in the cascade; paper: 10 of 6061)")
neg_mono = np.zeros((nb, nwin), np.float32)
for i in range(nb):
    ii, ii2 = scene_integrals(neg_imgs[i])
    neg_mono[i], _, _ = cascade_scan([mono], ii, ii2, wins, W, H)
neg_base = {k: np.zeros((nb, nwin), np.float32) for k in baselines}
for i in range(nb if baselines else 0):
    X = standardize(extract_patches(neg_imgs[i], wins))
    for k, (fn, _, _) in baselines.items():
        neg_base[k][i] = batched(fn, X)

### Detection curves

In [ ]:
methods = {  # name -> (scores of test cats, scores of dog windows, native threshold, fit seconds)
    "Viola-Jones cascade": (keys_24(stages, *Pte), neg_vj[:nb], float(K), t_casc),
    f"Single AdaBoost ({a.mono_features} feat)": (keys_24([mono], *Pte), neg_mono, 1.0, t_mono),
}
for k, (fn, th, tf) in baselines.items():
    methods[k] = (batched(fn, Xpos), neg_base[k], th, tf)

LEVELS = np.array([1e-2, 1e-3, 1e-4, 1e-5, 1e-6])
rows, curves = [], {}

def add_row(name, pos_s, neg_s, th, tf):
    fpr, tpr = roc_from_scores(pos_s, neg_s.ravel())
    rows.append(dict(name=name, neg_windows=int(neg_s.size), fit_s=tf,
                     det_at_fpr=[float(v) for v in np.interp(LEVELS, fpr, tpr, left=np.nan, right=np.nan)],
                     op_detection=float((pos_s >= th).mean()), op_fpr=float((neg_s >= th).mean())))
    curves[name] = (fpr, tpr)

for name, (p_, n_, th, tf) in methods.items():
    add_row(name, p_, n_, th, tf)
if ni > nb:  # the cascade on all scanned photos
    add_row(f"Viola-Jones cascade (all {ni} photos)", methods["Viola-Jones cascade"][0], neg_vj, float(K), t_casc)

### Speed

In [ ]:
# ---- speed (paper's speed setting: scale 1.25, step 1.5) ----------------------------------
wins_s = build_windows(W, H, a.speed_scale0, a.factor, a.speed_delta)
nws = sum(len(w["x"]) * len(w["y"]) for w in wins_s)
idx = range(min(a.speed_images, ni))

def timeit(fn):
    t = time.perf_counter()
    for i in idx:
        fn(neg_imgs[i])
    return (time.perf_counter() - t) / len(idx)

def run_cascade(stg):
    def f(img):
        ii, ii2 = scene_integrals(img)
        return cascade_scan(stg, ii, ii2, wins_s, W, H)
    return f

speed = [dict(name="Viola-Jones cascade", sec_per_image=timeit(run_cascade(stages))),
         dict(name=f"Single AdaBoost ({a.mono_features} feat)", sec_per_image=timeit(run_cascade([mono])))]
for k, (fn, _, _) in baselines.items():  # time includes image pyramid + window extraction + standardisation
    speed.append(dict(name=k, sec_per_image=timeit(lambda img, fn=fn: batched(fn, standardize(extract_patches(img, wins_s))))))
ii, ii2 = scene_integrals(neg_imgs[0])
_, _, nfs = cascade_scan(stages, ii, ii2, wins_s, W, H)
for r in speed:
    r["windows_per_s"] = nws / r["sec_per_image"]
speed[0]["features_per_window"] = nfs / nws
speed[1]["features_per_window"] = float(a.mono_features)

### Report

In [ ]:
# ---- report ---------------------------------------------------------------------------------
print(f"\nDETECTION RATE (cats found) at fixed false-positive rates. FP rate = false positives / windows scanned "
      f"({nwin:,} windows per photo, so FP rate 1e-4 = {1e-4 * nwin:.1f} false positives per photo).")
hdr = f"{'method':44s} {'neg win':>9s} " + " ".join(f"{'FPR=' + format(l, '.0e'):>9s}" for l in LEVELS) + f" {'op det':>7s} {'op FPR':>9s} {'fit s':>7s}"
print(hdr + "\n" + "-" * len(hdr))
fmt = lambda v: "    -    " if np.isnan(v) else f"{100 * v:8.1f}%"
for r in rows:
    print(f"{r['name']:44s} {r['neg_windows']:9,d} " + " ".join(fmt(v) for v in r["det_at_fpr"]) +
          f" {100 * r['op_detection']:6.1f}% {r['op_fpr']:9.2e} {r['fit_s']:7.0f}")
print("('-' = that false-positive rate is not reachable.  op = each method's own native threshold; the cascade's is "
      "'passes every stage'.)")
print(f"\nSPEED (scale {a.speed_scale0}, step {a.speed_delta}, {nws:,} windows per {W}x{H} photo, {len(idx)} photos)")
for r in speed:
    fw = f"  {r['features_per_window']:.1f} features/window" if "features_per_window" in r else ""
    print(f"{r['name']:44s} {r['sec_per_image']:8.3f} s/photo {r['windows_per_s']:12,.0f} windows/s{fw}")
print(f"\ncascade vs single AdaBoost classifier: {speed[1]['sec_per_image'] / speed[0]['sec_per_image']:.1f}x faster "
      f"(paper: the cascade is ~15x faster than the fastest previous system)")
print("Note: baselines include image-pyramid patch extraction in their time; the cascade scales the detector instead.")

json.dump(dict(args=vars(a), stage_sizes=sizes, total_features=sum(sizes), windows_per_photo=nwin,
               detection=rows, speed=speed), open(os.path.join(a.out, "results.json"), "w"), indent=2)

### Save and plot

In [ ]:
make_plots(a.out, curves, speed)
log(f"wrote results.json and results.png to {os.path.abspath(a.out)}")

from IPython.display import Image as IPImage, display
display(IPImage(filename=os.path.join(a.out, "results.png")))